# Cloud Provider Analytics · Notebook 02 — Pipeline end-to-end (Landing → Bronze → Silver → Gold → AstraDB)

**Big Data · ITBA · 2C 2026 · Segunda entrega (implementación técnica)**
Grupo: Cayetana Peces · Abril Solé · Agustina Edalian · Delfina Vázquez Escati

El notebook orquesta el pipeline; la lógica está en `src/` (funciones chicas y testeadas en `tests/`) y la configuración en `config/pipeline.example.yaml`.

| Sección | Qué hace | Responsable |
|---|---|---|
| 0. Setup | Repo, configuración, SparkSession | Delfina |
| 1. Landing | Descompresión e inventario (solo lectura) | Cayetana |
| 2. Bronze batch | 7 maestros con esquema explícito, quarantine y dedupe | Cayetana |
| 3. Bronze streaming | Structured Streaming: watermark, dedupe por `event_id`, late data, checkpoint | Cayetana |
| 4. Silver | `dim_org` + eventos conformados, reglas de calidad, quarantine | Abril |
| 5. Analítica | Anomalías de costo con MAD por (servicio, métrica) | Abril |
| 6. Gold | Mart `org_daily_usage_by_service` + `top_services_14d` | Agustina |
| 7. Idempotencia | Re-ejecución sin duplicados (conteos antes/después) | Delfina |
| 8. Serving | Carga a AstraDB y consultas Q1 y Q2 | Agustina |
| 9. Evidencias | run_log, quarantine, rutas, tamaños | Delfina |

**Para correrlo en Colab:** ver el Quickstart del README (subir el dataset y el bundle de AstraDB, cargar el token en *Secrets* y *Ejecutar todo*).

## 0. Setup

In [1]:
import os, sys

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/delfivazqueze/cloud-provider-analytics.git"

if EN_COLAB:
    !pip -q install pyspark==3.5.3 cassandra-driver pyyaml setuptools
    if not os.path.exists("/content/cloud-provider-analytics"):
        !git clone -q {REPO_URL} /content/cloud-provider-analytics
    REPO = "/content/cloud-provider-analytics"
else:
    REPO = os.path.abspath("..")          # corriendo desde notebooks/ del repo
sys.path.insert(0, REPO)

import pyspark
print("PySpark", pyspark.__version__)

PySpark 3.5.3


In [2]:
import glob, json, time, zipfile, shutil
from pyspark.sql import SparkSession, functions as F

from src.common.config import load_config
from src.common.run_log import RunLog, new_run_id

cfg = load_config(REPO, base="/content")
P = cfg["paths"]

spark = (SparkSession.builder
         .appName(cfg["spark"]["app_name"])
         .config("spark.sql.shuffle.partitions", cfg["spark"]["shuffle_partitions"])
         .config("spark.sql.sources.partitionOverwriteMode", cfg["spark"]["partition_overwrite_mode"])
         .config("spark.sql.session.timeZone", cfg["spark"]["timezone"])
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RUN_ID = new_run_id()
log = RunLog(spark, P["meta"], RUN_ID)
os.makedirs(P["evidence"], exist_ok=True)
print("Spark", spark.version, "| run_id:", RUN_ID)
print(json.dumps({k: v for k, v in cfg.items() if k != "paths"}, indent=1))

Spark 3.5.3 | run_id: 20261109T000406-e50d16
{
 "spark": {
  "app_name": "cloud-provider-analytics",
  "shuffle_partitions": 8,
  "partition_overwrite_mode": "dynamic",
  "timezone": "UTC"
 },
 "streaming": {
  "max_files_per_trigger": 30,
  "watermark": "60 days",
  "trigger": "available_now",
  "processing_time": "5 minutes"
 },
 "quality": {
  "cost_tolerance_usd": -0.01,
  "anomaly_method": "mad_by_service_metric",
  "anomaly_threshold": 3.5,
  "nps_min": -100,
  "nps_max": 100
 },
 "cassandra": {
  "keyspace": "cloud_analytics",
  "secure_bundle_env": "ASTRA_SCB_PATH",
  "token_env": "ASTRA_DB_APPLICATION_TOKEN",
  "concurrency": 50
 }
}


## 1. Landing

Landing es inmutable: se descomprime una sola vez y después solo se lee. Para empezar de cero (demo) alcanza con borrar `datalake/` salvo `landing/` (celda de limpieza al final).

In [3]:
zip_path = f"{P['base']}/cloud_provider_challenge_dataset_v1.zip"
if not os.path.exists(f"{P['landing']}/customers_orgs.csv"):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(P["base"])
archivos_eventos = sorted(glob.glob(f"{P['landing']}/usage_events_stream/*.jsonl"))
print("CSV en landing:", sorted(os.path.basename(f) for f in glob.glob(f"{P['landing']}/*.csv")))
print("Archivos de eventos:", len(archivos_eventos))

CSV en landing: ['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'users.csv']
Archivos de eventos: 120


## 2. Bronze batch: maestros y facturación

Se ingestan los 7 CSV (la consigna pide al menos 3). Para cada uno:
- esquema explícito (`src/common/schemas.py`) y modo `PERMISSIVE` con columna `_corrupt_record`: una fila mal formada no rompe la carga, va a quarantine (`B01`);
- clave natural nula → quarantine (`B02`); duplicados por clave natural → se descartan;
- columnas técnicas `ingest_ts`, `ingest_date`, `source_file`, `run_id`;
- partición por `ingest_date` (snapshot del día) y `billing_month` para facturación. El overwrite dinámico reemplaza solo la partición del día, así que re-ejecutar no duplica.

In [4]:
from src.common.schemas import CSV_SCHEMAS
from src.jobs.bronze_batch import ingest_master

print(f"{'maestro':<20}{'entrada':>9}{'bronze':>9}{'quarantine':>12}")
for name in CSV_SCHEMAS:
    n_in, n_ok, n_q = ingest_master(spark, cfg, name, RUN_ID, log)
    print(f"{name:<20}{n_in:>9,}{n_ok:>9,}{n_q:>12,}")
log.flush()

maestro               entrada   bronze  quarantine


customers_orgs             80       80           0


users                     800      800           0


resources                 400      400           0


support_tickets         1,000    1,000           0


marketing_touches       1,500    1,500           0


nps_surveys                92       92           0


billing_monthly           240      240           0


> **Hallazgo de la regla B01.** En la primera corrida la regla mandó 211 filas de `resources` a quarantine como corruptas. El motivo: `tags_json` trae comillas escapadas al estilo CSV estándar (`"[""env:prod""]"`) y Spark, por defecto, escapa con `\`. Se agregó `escape='"'` a la lectura y desde entonces hay 0 filas corruptas. En la E1 (sin columna `_corrupt_record`) esas filas se leían con el campo truncado sin que nadie se enterara.

## 3. Bronze streaming: `usage_events_stream/*.jsonl`

### 3.1 Elegir el watermark con datos

El watermark es `máx(event_time visto) − tolerancia`, y todo evento más viejo que eso se descarta por tardío. En la E1 medimos que **cada archivo trae eventos de los 60 días** (`evidence/01_event_time_range_by_file.csv`). Antes de fijar el valor, corremos la misma lectura + dedupe con dos tolerancias contra un sink `noop` (no escribe nada) y medimos `numRowsDroppedByWatermark`.

In [5]:
from src.jobs.bronze_stream import watermark_experiment

experimento = []
for wm in ["2 days", "60 days"]:
    r = watermark_experiment(spark, cfg, wm)
    r["watermark"] = wm
    experimento.append(r)
    print(r)

{'micro_lotes': 4, 'filas_leidas': 43200, 'descartadas_por_watermark': 20874, 'emitidas': 22326, 'claves_en_estado': 2181, 'watermark': '2 days'}


{'micro_lotes': 4, 'filas_leidas': 43200, 'descartadas_por_watermark': 0, 'emitidas': 43200, 'claves_en_estado': 43200, 'watermark': '60 days'}


Con 2 días, a partir del segundo micro-lote el watermark queda cerca del 29/08 y **se pierde casi la mitad de los eventos** (todo lo que llega en los lotes 2 a 4 con fecha anterior). Con **60 días** no se pierde ningún evento y el estado del dedupe son ~43 mil `event_id` (chico). Se fija `watermark = "60 days"` (D15). En un caso real con eventos ordenados se usaría una tolerancia corta y el recálculo batch de Lambda corrige lo que llegue más tarde.

### 3.2 Ingesta en streaming

- `readStream` con esquema explícito y `maxFilesPerTrigger = 30` (admission control): 120 archivos → 4 micro-lotes.
- Reglas de Bronze: `event_id` no nulo (`B02`), timestamp parseable (`B03`), `schema_version ∈ {1,2}` (`B04`). Los inválidos van por una **segunda query** a quarantine.
- `withWatermark("event_ts", "60 days")` + `dropDuplicatesWithinWatermark(["event_id"])`.
- Sink Parquet particionado por `event_date`, `outputMode("append")` y **checkpoint propio por query**.
- `trigger(availableNow=True)`: procesa todo lo pendiente en micro-lotes y se detiene (en producción se cambia a `processingTime="5 minutes"` desde la config, mismo código).

In [6]:
from src.jobs.bronze_stream import start_bronze_queries, progress_summary

t0 = time.time()
q_bronze, q_quar = start_bronze_queries(spark, cfg, RUN_ID)
q_bronze.awaitTermination(); q_quar.awaitTermination()
dur = time.time() - t0

prog_bronze = progress_summary(q_bronze)
print("bronze_usage_events :", prog_bronze)

def hay_parquet(path):
    return len(glob.glob(f"{path}/**/*.parquet", recursive=True)) > 0

bronze_ev = spark.read.parquet(f"{P['bronze']}/usage_events")
n_bronze = bronze_ev.count()
n_q_bronze = (spark.read.parquet(f"{P['quarantine']}/usage_events_bronze").count()
              if hay_parquet(f"{P['quarantine']}/usage_events_bronze") else 0)
print(f"Bronze usage_events: {n_bronze:,} filas | quarantine Bronze: {n_q_bronze:,}")
log.add("bronze_stream", "bronze", "usage_events", prog_bronze["filas_leidas"], n_bronze, n_q_bronze,
        prog_bronze["filas_leidas"] - n_q_bronze - n_bronze, dur,
        detalle=f"micro_lotes={prog_bronze['micro_lotes']}; tarde={prog_bronze['descartadas_por_watermark']}")
log.flush()

bronze_usage_events : {'micro_lotes': 4, 'filas_leidas': 43200, 'descartadas_por_watermark': 0, 'emitidas': 43200, 'claves_en_estado': 43200}


Bronze usage_events: 43,200 filas | quarantine Bronze: 0


In [7]:
# Métricas del último micro-lote (lo que se miraría en producción para alertar)
lp = q_bronze.lastProgress
print(json.dumps({"batchId": lp["batchId"], "numInputRows": lp["numInputRows"],
                  "durationMs": lp["durationMs"], "stateOperators": lp["stateOperators"],
                  "sources": [{k: s[k] for k in ("description", "startOffset", "endOffset")} for s in lp["sources"]]},
                 indent=1, default=str)[:2500])
print("\nCheckpoint:", sorted(os.listdir(f"{P['checkpoints']}/bronze_usage_events")))

{
 "batchId": 4,
 "numInputRows": 0,
 "durationMs": {
  "addBatch": 604,
  "commitOffsets": 40,
  "getBatch": 0,
  "latestOffset": 0,
  "queryPlanning": 27,
  "triggerExecution": 708,
  "walCommit": 33
 },
 "stateOperators": [
  {
   "operatorName": "dedupeWithinWatermark",
   "numRowsTotal": 43200,
   "numRowsUpdated": 0,
   "allUpdatesTimeMs": 21,
   "numRowsRemoved": 0,
   "allRemovalsTimeMs": 0,
   "commitTimeMs": 428,
   "memoryUsedBytes": 11321728,
   "numRowsDroppedByWatermark": 0,
   "numShufflePartitions": 8,
   "numStateStoreInstances": 8,
   "customMetrics": {
    "loadedMapCacheHitCount": 32,
    "loadedMapCacheMissCount": 0,
    "numDroppedDuplicateRows": 0,
    "stateOnCurrentVersionSizeBytes": 7869504
   }
  }
 ],
 "sources": [
  {
   "description": "FileStreamSource[file:/content/datalake/landing/usage_events_stream]",
   "startOffset": {
    "logOffset": 3
   },
   "endOffset": {
    "logOffset": 3
   }
  }
 ]
}

Checkpoint: ['.metadata.crc', 'commits', 'metadata', 'of

## 4. Silver

### 4.1 `dim_org` (maestro conformado)
Último snapshot de Bronze (SCD tipo 1 en el MVP), tipos explícitos (`is_enterprise` booleano, `signup_date` fecha), strings normalizados y NPS fuera de [−100, 100] → `null` + flag `is_nps_out_of_range`.

In [8]:
from src.jobs.silver import build_dim_org, build_silver_events

dim_org = build_dim_org(spark, cfg, RUN_ID, log)
dim_org.printSchema()
dim_org.filter("is_nps_out_of_range").select("org_id", "org_name", "nps_score", "is_nps_out_of_range").show()

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- is_nps_out_of_range: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- updated_ts: timestamp (nullable = true)
 |-- run_id: string (nullable = true)



+------------+--------------+---------+-------------------+
|      org_id|      org_name|nps_score|is_nps_out_of_range|
+------------+--------------+---------+-------------------+
|org_pac56t4u|Gamma Cloud 37|     NULL|               true|
+------------+--------------+---------+-------------------+



### 4.2 Eventos conformados

Pasos (`src/jobs/silver.py`):
1. Normalización de `service`, `region`, `metric`, `unit` (minúsculas, sin espacios).
2. **v1/v2 compatibles:** `carbon_kg` y `genai_tokens` quedan `null` en v1 (no 0: "no medido" ≠ "cero").
3. **Regla R3 — `unit` no nulo cuando hay `value`:** se imputa desde `metric` (relación 1:1 medida en la E1) y queda el flag `unit_imputed`. Si la métrica no es conocida → quarantine `S02`.
4. **Regla R2 — `cost_usd_increment >= -0.01`:** menor a −0.01 → quarantine `S01` con `is_cost_anomaly = true`; entre −0.01 y 0 se lleva a 0 (`is_cost_adjusted`).
5. **Join de enriquecimiento** con `dim_org` (broadcast: 80 filas) y regla `S03` (org inexistente).
6. Features a nivel evento: `requests`, `cpu_hours`, `storage_gb_hours`, `genai_tokens`, `carbon_kg`.
7. Score de anomalía (sección 5).

In [9]:
silver_ev = build_silver_events(spark, cfg, dim_org, RUN_ID, log)
log.flush()

quar_silver = spark.read.parquet(f"{P['quarantine']}/usage_events_silver")
print(f"Bronze: {n_bronze:,} | Silver: {silver_ev.count():,} | Quarantine Silver: {quar_silver.count():,}")
print("Conciliación (O2) → Bronze == Silver + Quarantine:", n_bronze == silver_ev.count() + quar_silver.count())
quar_silver.groupBy("rule_id", "motivo").count().show(truncate=False)

Bronze: 43,200 | Silver: 42,989 | Quarantine Silver: 211
Conciliación (O2) → Bronze == Silver + Quarantine: True


+------------------+------------------------------------------------------------+-----+
|rule_id           |motivo                                                      |count|
+------------------+------------------------------------------------------------+-----+
|S01_COSTO_NEGATIVO|cost_usd_increment < -0.01 (se marca además is_cost_anomaly)|211  |
+------------------+------------------------------------------------------------+-----+



In [10]:
# Compatibilidad de esquema (O4): todos los v1 y v2 de Bronze están en Silver o explicados en quarantine
b = bronze_ev.groupBy("schema_version").count().withColumnRenamed("count", "bronze")
s = silver_ev.groupBy("schema_version").agg(F.count("*").alias("silver"),
        F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon"),
        F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_tokens"))
qv = quar_silver.groupBy("schema_version").count().withColumnRenamed("count", "quarantine")
b.join(s, "schema_version").join(qv, "schema_version", "left").orderBy("schema_version").show()

# Regla R3: cuántas unit se imputaron y por métrica
silver_ev.filter("unit_imputed").groupBy("metric", "unit").count().orderBy("metric").show()
print("Eventos con value y sin unit que quedan en Silver:",
      silver_ev.filter(F.col("value_num").isNotNull() & F.col("unit").isNull()).count())

+--------------+------+------+----------+----------+----------+
|schema_version|bronze|silver|con_carbon|con_tokens|quarantine|
+--------------+------+------+----------+----------+----------+
|             1| 10800| 10738|         0|         0|        62|
|             2| 32400| 32251|     32251|      3117|       149|
+--------------+------+------+----------+----------+----------+



+----------------+--------+-----+
|          metric|    unit|count|
+----------------+--------+-----+
|       cpu_hours|   hours|  500|
|        requests|   count|  896|
|storage_gb_hours|gb_hours|  637|
+----------------+--------+-----+



Eventos con value y sin unit que quedan en Silver: 0


In [11]:
# Muestras de quarantine (evidencia verificable de las reglas)
(quar_silver.select("rule_id", "event_id", "event_ts", "org_id", "service", "metric", "cost_usd_increment",
                    "is_cost_anomaly", "motivo", "source_file")
    .orderBy("cost_usd_increment").show(5, truncate=40))

+------------------+----------------+-------------------+------------+---------+--------+------------------+---------------+----------------------------------------+----------------------------------------+
|           rule_id|        event_id|           event_ts|      org_id|  service|  metric|cost_usd_increment|is_cost_anomaly|                                  motivo|                             source_file|
+------------------+----------------+-------------------+------------+---------+--------+------------------+---------------+----------------------------------------+----------------------------------------+
|S01_COSTO_NEGATIVO|evt_t5brta67j5l3|2025-08-24 13:41:00|org_0lvsnujz|analytics|requests|         -154.4608|           true|cost_usd_increment < -0.01 (se marca ...|file:///content/datalake/landing/usag...|
|S01_COSTO_NEGATIVO|evt_9ur456xoexl5|2025-07-28 04:25:00|org_h2m5zja8|    genai|requests|          -19.8245|           true|cost_usd_increment < -0.01 (se marca ...|file://

In [12]:
# El join con dim_org es un broadcast hash join (sin shuffle de los eventos)
from src.jobs.silver import conform_events
conform_events(bronze_ev, dim_org, cfg["quality"]["cost_tolerance_usd"]).explain(mode="simple")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [org_id#10409, event_id#10407, event_ts#10408-T5184000000ms, resource_id#10410, service#17684, region#17703, metric#17722, value_raw#10414, value_num#10415, unit#17780, cost_usd_increment#17821, schema_version#10418, carbon_kg#10419, genai_tokens#10420L, ingest_ts#10421, source_file#10422, run_id#10423, event_date#10424, unit_imputed#17760, is_cost_adjusted#17800, requests#17884, cpu_hours#17906, storage_gb_hours#17929, org_name#10627, ... 3 more fields]
   +- BroadcastHashJoin [org_id#10409], [org_id#10626], LeftOuter, BuildRight, false
      :- Project [event_id#10407, event_ts#10408-T5184000000ms, org_id#10409, resource_id#10410, service#17684, region#17703, metric#17722, value_raw#10414, value_num#10415, unit#17780, CASE WHEN is_cost_adjusted#17800 THEN 0.0 ELSE cost_usd_increment#10417 END AS cost_usd_increment#17821, schema_version#10418, carbon_kg#10419, genai_tokens#10420L, ingest_ts#10421, source_file#10422, ru

## 5. Componente analítico: anomalías de costo

No hay una variable objetivo etiquetada (nadie marcó qué costos son anómalos), así que en lugar de un modelo supervisado de Spark ML usamos un método **no supervisado y robusto** de los que menciona la consigna (z-score, MAD o percentiles), tal como quedó planteado en la E1 (D09):

`mad_score = 0.6745 · |costo − mediana| / MAD`, calculado **por (servicio, métrica)**, y `is_cost_anomaly = mad_score > 3.5`.

En la E1 vimos que con un umbral global el z-score marca 70 eventos y el MAD el 25 %: el costo depende del servicio y la métrica. Los parámetros de cada grupo quedan guardados en `_meta/anomaly_params` para poder auditar por qué un evento fue marcado.

In [13]:
params = spark.read.parquet(f"{P['meta']}/anomaly_params")
anom = (silver_ev.groupBy("service", "metric")
        .agg(F.count("*").alias("eventos"),
             F.sum(F.col("is_cost_anomaly").cast("int")).alias("anomalias"))
        .join(params.select("service", "metric", F.round("cost_median", 3).alias("mediana"),
                            F.round("cost_mad", 3).alias("mad")), ["service", "metric"])
        .withColumn("tasa_%", F.round(100 * F.col("anomalias") / F.col("eventos"), 2))
        .orderBy("service", "metric"))
anom.show(20)
anom_pd = anom.toPandas()

tot = silver_ev.agg(F.count("*").alias("n"), F.sum(F.col("is_cost_anomaly").cast("int")).alias("a")).collect()[0]
tasa = 100 * tot["a"] / tot["n"]
print(f"Anomalías: {tot['a']:,} de {tot['n']:,} eventos → {tasa:.2f} % (objetivo O7: revisar si supera ~1 %)")

+----------+----------------+-------+---------+-------+-----+------+
|   service|          metric|eventos|anomalias|mediana|  mad|tasa_%|
+----------+----------------+-------+---------+-------+-----+------+
| analytics|       cpu_hours|   1146|        6|  0.128|0.071|  0.52|
| analytics|        requests|   2063|        9|  8.359|0.956|  0.44|
| analytics|storage_gb_hours|   1362|        5|  0.706|0.383|  0.37|
|   compute|       cpu_hours|   3126|       10|  0.141|0.083|  0.32|
|   compute|        requests|   5545|       30|   9.53|1.172|  0.54|
|   compute|storage_gb_hours|   3772|       24|  0.771| 0.43|  0.64|
|  database|       cpu_hours|   1762|        7|  0.093| 0.05|   0.4|
|  database|        requests|   3333|       14|  5.964|0.707|  0.42|
|  database|storage_gb_hours|   2280|        9|    0.5|0.284|  0.39|
|     genai|       cpu_hours|   1072|        3|  0.207|0.125|  0.28|
|     genai|        requests|   1832|        6| 14.463|1.654|  0.33|
|     genai|storage_gb_hours|   12

Anomalías: 191 de 42,989 eventos → 0.44 % (objetivo O7: revisar si supera ~1 %)


In [14]:
# Contexto de las anomalías más fuertes (lo que vería un analista de FinOps)
(silver_ev.filter("is_cost_anomaly")
    .select("event_date", "org_name", "service", "metric", "value_num", "cost_usd_increment", "mad_score")
    .orderBy(F.desc("mad_score")).show(8, truncate=False))

# Comparación con el z-score global (el otro método evaluado en la E1) sobre los mismos datos de Silver
st = silver_ev.agg(F.mean("cost_usd_increment").alias("m"), F.stddev("cost_usd_increment").alias("s")).collect()[0]
comp = (silver_ev.withColumn("z_global_3", (F.abs((F.col("cost_usd_increment") - st["m"]) / st["s"]) > 3).cast("int"))
        .groupBy("service")
        .agg(F.sum("z_global_3").alias("z_score_global>3"),
             F.sum(F.col("is_cost_anomaly").cast("int")).alias("mad_por_grupo>3.5"))
        .orderBy("service"))
comp.show()

+----------+----------------+----------+--------+---------+------------------+---------+
|event_date|org_name        |service   |metric  |value_num|cost_usd_increment|mad_score|
+----------+----------------+----------+--------+---------+------------------+---------+
|2025-08-25|Nimbus Data 18  |analytics |requests|131.0    |249.3861          |170.019  |
|2025-07-09|Apex Cloud 34   |compute   |requests|133.0    |302.3459          |168.577  |
|2025-08-29|Nova Cloud 45   |analytics |requests|128.0    |221.2907          |150.201  |
|2025-07-09|Omega Digital 40|compute   |requests|151.0    |261.3299          |144.964  |
|2025-08-20|Delta Digital 10|storage   |requests|124.0    |62.9922           |143.098  |
|2025-07-21|Delta Tech 52   |analytics |requests|127.0    |209.8516          |142.132  |
|2025-07-23|Alpha Labs 21   |database  |requests|125.0    |149.9091          |137.406  |
|2025-07-11|Gamma Labs 73   |networking|requests|125.0    |29.6319           |132.605  |
+----------+---------

+----------+----------------+-----------------+
|   service|z_score_global>3|mad_por_grupo>3.5|
+----------+----------------+-----------------+
| analytics|               9|               20|
|   compute|              30|               64|
|  database|              11|               30|
|     genai|               5|               15|
|networking|               2|               27|
|   storage|              13|               35|
+----------+----------------+-----------------+



**Lectura de resultados.** Se marcan 191 eventos (0,44 %), por debajo del ~1 % que nos pusimos como techo en O7, y las alertas se reparten entre todos los servicios (entre 0,3 % y 0,7 % por grupo). El z-score global, en cambio, depende de la escala del costo: en `networking`, que es barato, marca solo 2 eventos, porque un spike de 30 USD queda "normal" comparado con los costos de compute o GenAI. Las anomalías más fuertes son requests con costo 100 veces por encima de la mediana de su grupo, que es justo lo que FinOps quiere revisar. **Limitaciones:** no hay etiquetas para medir precisión/recall; el umbral 3.5 es el valor estándar del método (Iglewicz y Hoaglin) y no está calibrado con feedback de negocio. **Plan de cierre (final):** `cost_anomaly_mart` con grano (org, fecha, servicio) y evaluar un modelo de Spark ML solo si se consigue una variable objetivo (ver `docs/BACKLOG.md`).

## 6. Gold: mart FinOps `org_daily_usage_by_service`

Grano: **(org_id, usage_date, service)**. Es la misma agregación que validamos con MapReduce en la E1, ahora desde Silver (datos limpios). Se escribe particionado por `usage_date`.
Además se precalcula `top_services_14d` (ranking de servicios por costo de los últimos 14 días para cada org y fecha), porque Cassandra no puede ordenar por un agregado (D13).

In [15]:
from src.jobs.gold import build_gold

mart, top = build_gold(spark, cfg, RUN_ID, log)
log.flush()
mart.printSchema()

dup_grano = mart.groupBy("org_id", "usage_date", "service").count().filter("count > 1").count()
cost_silver = silver_ev.agg(F.round(F.sum("cost_usd_increment"), 2)).collect()[0][0]
cost_gold = mart.agg(F.round(F.sum("daily_cost_usd"), 2)).collect()[0][0]
print(f"Filas del mart: {mart.count():,} | claves duplicadas: {dup_grano}")
print(f"Conciliación de costos → Silver: {cost_silver:,.2f} USD | Gold: {cost_gold:,.2f} USD | "
      f"diferencia: {abs(cost_silver - cost_gold):.2f} (redondeo a 4 decimales por fila)")

root
 |-- org_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- daily_cost_usd: double (nullable = true)
 |-- requests: double (nullable = true)
 |-- cpu_hours: double (nullable = true)
 |-- storage_gb_hours: double (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- events: long (nullable = true)
 |-- anomaly_events: long (nullable = true)
 |-- run_id: string (nullable = true)
 |-- usage_date: date (nullable = true)



Filas del mart: 11,050 | claves duplicadas: 0
Conciliación de costos → Silver: 148,351.60 USD | Gold: 148,351.60 USD | diferencia: 0.00 (redondeo a 4 decimales por fila)


In [16]:
(mart.orderBy(F.desc("daily_cost_usd"))
     .select("org_id", "usage_date", "service", "daily_cost_usd", "requests", "cpu_hours",
             "storage_gb_hours", "genai_tokens", "carbon_kg", "events", "anomaly_events")
     .show(5))

# Org de ejemplo para las consultas: la de mayor costo en los últimos 14 días
ultima_fecha = mart.agg(F.max("usage_date")).collect()[0][0]
ORG_EJEMPLO = (top.filter(F.col("as_of_date") == ultima_fecha).filter("rank = 1")
                  .orderBy(F.desc("cost_14d_usd")).first()["org_id"])
print("Última fecha:", ultima_fecha, "| org de ejemplo:", ORG_EJEMPLO)
top.filter((F.col("org_id") == ORG_EJEMPLO) & (F.col("as_of_date") == ultima_fecha)).orderBy("rank").show()

+------------+----------+-------+--------------+--------+---------+----------------+------------+---------+------+--------------+
|      org_id|usage_date|service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|events|anomaly_events|
+------------+----------+-------+--------------+--------+---------+----------------+------------+---------+------+--------------+
|org_rixa11dp|2025-08-24|  genai|      340.5259|   390.0|   0.0473|          8.3086|        5646|   0.0797|     6|             1|
|org_c11ertj5|2025-07-11|  genai|      334.6767|   237.0|     NULL|         12.9349|        NULL|     NULL|     3|             1|
|org_tvhhpbmy|2025-07-04|  genai|      313.4168|   210.0|   1.3851|            NULL|        NULL|     NULL|     3|             1|
|org_x7eedtjv|2025-07-09|compute|       311.255|   249.0|   1.8107|            NULL|        NULL|     NULL|     3|             1|
|org_5935a0l7|2025-07-09|compute|      271.6404|   267.0|   1.1745|          7.2181|      

Última fecha: 2025-08-31 | org de ejemplo: org_ktakpuxq
+------------+----------+----+----------+------------+------------+--------------------+
|      org_id|as_of_date|rank|   service|cost_14d_usd|requests_14d|              run_id|
+------------+----------+----+----------+------------+------------+--------------------+
|org_ktakpuxq|2025-08-31|   1|   compute|    651.5958|      7256.0|20261109T000406-e...|
|org_ktakpuxq|2025-08-31|   2|     genai|    141.8207|      1115.0|20261109T000406-e...|
|org_ktakpuxq|2025-08-31|   3|  database|     64.3944|      1296.0|20261109T000406-e...|
|org_ktakpuxq|2025-08-31|   4|networking|      8.2631|       733.0|20261109T000406-e...|
+------------+----------+----+----------+------------+------------+--------------------+



## 7. Idempotencia

1. **Streaming:** se vuelve a lanzar la misma query; el checkpoint ya registra los 120 archivos como procesados, así que no lee nada nuevo.
2. **Batch (Bronze, Silver, Gold):** se re-ejecutan los jobs; el dedupe por clave natural + overwrite dinámico por partición dejan los mismos conteos.

In [17]:
def conteos():
    return {
        "bronze.usage_events": spark.read.parquet(f"{P['bronze']}/usage_events").count(),
        "bronze.customers_orgs": spark.read.parquet(f"{P['bronze']}/customers_orgs").count(),
        "bronze.users": spark.read.parquet(f"{P['bronze']}/users").count(),
        "bronze.billing_monthly": spark.read.parquet(f"{P['bronze']}/billing_monthly").count(),
        "silver.usage_events": spark.read.parquet(f"{P['silver']}/usage_events").count(),
        "silver.dim_org": spark.read.parquet(f"{P['silver']}/dim_org").count(),
        "quarantine.usage_events_silver": spark.read.parquet(f"{P['quarantine']}/usage_events_silver").count(),
        "gold.org_daily_usage_by_service": spark.read.parquet(f"{P['gold']}/org_daily_usage_by_service").count(),
        "gold.top_services_14d": spark.read.parquet(f"{P['gold']}/top_services_14d").count(),
    }

antes = conteos()

# 1) streaming: misma query, mismo checkpoint
q1, q2 = start_bronze_queries(spark, cfg, RUN_ID)
q1.awaitTermination(); q2.awaitTermination()
print("Re-ejecución del stream →", progress_summary(q1))

# 2) batch completo otra vez
log_re = RunLog(spark, P["meta"], RUN_ID + "-reejecucion")
for name in CSV_SCHEMAS:
    ingest_master(spark, cfg, name, log_re.run_id, log_re)
dim_org = build_dim_org(spark, cfg, log_re.run_id, log_re)
build_silver_events(spark, cfg, dim_org, log_re.run_id, log_re)
build_gold(spark, cfg, log_re.run_id, log_re)
log_re.flush()

despues = conteos()
idem = [(k, antes[k], despues[k], antes[k] == despues[k]) for k in antes]
print(f"\n{'dataset':<34}{'antes':>9}{'después':>9}  igual")
for k, a, d, ok in idem:
    print(f"{k:<34}{a:>9,}{d:>9,}  {ok}")
print("\nTodo idéntico:", all(x[3] for x in idem))

# los DataFrames de las secciones anteriores apuntan a archivos que la re-ejecución reemplazó: se releen
silver_ev = spark.read.parquet(f"{P['silver']}/usage_events")
quar_silver = spark.read.parquet(f"{P['quarantine']}/usage_events_silver")
mart = spark.read.parquet(f"{P['gold']}/org_daily_usage_by_service")
top = spark.read.parquet(f"{P['gold']}/top_services_14d")

Re-ejecución del stream → {'micro_lotes': 0, 'filas_leidas': 0, 'descartadas_por_watermark': 0, 'emitidas': 0, 'claves_en_estado': 0}



dataset                               antes  después  igual
bronze.usage_events                  43,200   43,200  True
bronze.customers_orgs                    80       80  True
bronze.users                            800      800  True
bronze.billing_monthly                  240      240  True
silver.usage_events                  42,989   42,989  True
silver.dim_org                           80       80  True
quarantine.usage_events_silver          211      211  True
gold.org_daily_usage_by_service      11,050   11,050  True
gold.top_services_14d                12,314   12,314  True

Todo idéntico: True


## 8. Serving en Cassandra / AstraDB

Modelo **query-first** (`cql/01_schema.cql`): una tabla por consulta.

| Consulta | Tabla | Partition key | Clustering | Por qué |
|---|---|---|---|---|
| Q1 costos y requests diarios por org y servicio en un rango | `org_daily_usage_by_service` | `org_id` | `usage_date DESC, service` | El rango de fechas se resuelve dentro de una sola partición |
| Q2 Top-N servicios por costo de los últimos 14 días | `top_services_14d` | `(org_id, as_of_date)` | `rank ASC` | Cassandra no ordena por un agregado: el ranking viene precalculado y `LIMIT N` alcanza |

La carga usa el driver de Python con sentencias preparadas y escritura concurrente. Como `INSERT` en Cassandra es un **upsert por PRIMARY KEY**, recargar no duplica.

Credenciales: el token se lee de los *Secrets* de Colab (`ASTRA_DB_APPLICATION_TOKEN`) y el bundle de `/content/secure-connect-big-data.zip`. Nada de esto está en el repo.

In [18]:
SCB_PATH = f"{P['base']}/secure-connect-big-data.zip"
try:
    from google.colab import userdata
    TOKEN = userdata.get(cfg["cassandra"]["token_env"])
except Exception:
    TOKEN = os.environ.get(cfg["cassandra"]["token_env"])

HAY_ASTRA = bool(TOKEN) and os.path.exists(SCB_PATH)
print("Credenciales de AstraDB disponibles:", HAY_ASTRA)
if not HAY_ASTRA:
    print("Sin token o sin bundle: se saltea la sección 8 (el resto del pipeline no depende de ella).")

Credenciales de AstraDB disponibles: False
Sin token o sin bundle: se saltea la sección 8 (el resto del pipeline no depende de ella).


In [19]:
if HAY_ASTRA:
    from src.jobs.serving import connect, run_cql_file, load_dataframe, count_rows
    cluster, session = connect(SCB_PATH, TOKEN, cfg["cassandra"]["keyspace"])
    print("Conectado a", session.execute("SELECT release_version FROM system.local").one()[0],
          "| keyspace:", session.keyspace)
    print("Sentencias ejecutadas de cql/01_schema.cql:", run_cql_file(session, f"{REPO}/cql/01_schema.cql"))

In [20]:
if HAY_ASTRA:
    MART_CQL = ["org_id", "usage_date", "service", "org_name", "daily_cost_usd", "requests", "cpu_hours",
                "storage_gb_hours", "genai_tokens", "carbon_kg", "events", "anomaly_events", "run_id"]
    TOP_CQL = ["org_id", "as_of_date", "rank", "service", "cost_14d_usd", "requests_14d", "run_id"]
    mart_c = (mart.withColumn("events", F.col("events").cast("int"))
                  .withColumn("anomaly_events", F.col("anomaly_events").cast("int")))
    orgs = [r[0] for r in mart.select("org_id").distinct().collect()]

    t0 = time.time()
    enviadas, errores = load_dataframe(session, mart_c, "org_daily_usage_by_service", MART_CQL,
                                       cfg["cassandra"]["concurrency"])
    print(f"org_daily_usage_by_service → enviadas: {enviadas:,} | errores: {errores} | {time.time()-t0:.0f} s")
    t0 = time.time()
    enviadas_top, errores_top = load_dataframe(session, top, "top_services_14d", TOP_CQL,
                                               cfg["cassandra"]["concurrency"])
    print(f"top_services_14d          → enviadas: {enviadas_top:,} | errores: {errores_top} | {time.time()-t0:.0f} s")

    filas_cassandra = count_rows(session, "org_daily_usage_by_service", orgs)
    print(f"Filas en Cassandra (org_daily_usage_by_service): {filas_cassandra:,} | en Gold: {mart.count():,}")
    log.add("serving_cassandra", "serving", "org_daily_usage_by_service", enviadas, filas_cassandra,
            conciliar=False, detalle=f"errores={errores}; top_services_14d={enviadas_top}")
    log.flush()

In [21]:
if HAY_ASTRA:
    import pandas as pd
    q1 = f'''SELECT usage_date, service, daily_cost_usd, requests, events, anomaly_events
FROM org_daily_usage_by_service
WHERE org_id = '{ORG_EJEMPLO}'
  AND usage_date >= '2025-08-18' AND usage_date <= '2025-08-31';'''
    print("Q1 · CQL:\n" + q1)
    res_q1 = pd.DataFrame(list(session.execute(q1.rstrip(";"))))
    print(f"\n{len(res_q1)} filas")
    display(res_q1.head(15))

In [22]:
if HAY_ASTRA:
    q2 = f'''SELECT rank, service, cost_14d_usd, requests_14d
FROM top_services_14d
WHERE org_id = '{ORG_EJEMPLO}' AND as_of_date = '{ultima_fecha}'
LIMIT 3;'''
    print("Q2 · CQL:\n" + q2)
    res_q2 = pd.DataFrame(list(session.execute(q2.rstrip(";"))))
    display(res_q2)

    # Verificación cruzada: lo que devuelve Cassandra coincide con Gold
    esperado = (top.filter((F.col("org_id") == ORG_EJEMPLO) & (F.col("as_of_date") == ultima_fecha))
                   .orderBy("rank").limit(3).select("service").toPandas()["service"].tolist())
    print("Coincide con Gold:", esperado == res_q2["service"].tolist())

In [23]:
if HAY_ASTRA:
    # Idempotencia en serving: recargar el mismo mart no cambia la cantidad de filas (upsert por PK)
    antes_c = count_rows(session, "org_daily_usage_by_service", orgs)
    load_dataframe(session, mart_c, "org_daily_usage_by_service", MART_CQL, cfg["cassandra"]["concurrency"])
    despues_c = count_rows(session, "org_daily_usage_by_service", orgs)
    print(f"Cassandra antes: {antes_c:,} | después de recargar: {despues_c:,} | idéntico: {antes_c == despues_c}")
    res_q1.to_csv(f"{P['evidence']}/02_cassandra_q1.csv", index=False)
    res_q2.to_csv(f"{P['evidence']}/02_cassandra_q2.csv", index=False)
    cluster.shutdown()

## 9. Evidencias: run_log, rutas, tamaños y quarantine

In [24]:
run_log = log.read()
(run_log.filter(F.col("run_id") == RUN_ID)
    .select("job", "dataset", "filas_entrada", "filas_validas", "filas_quarantine", "filas_duplicadas",
            "duracion_s", "estado", "detalle")
    .orderBy("ts").show(30, truncate=60))
print("Ejecuciones con conciliación fallida:", run_log.filter("estado <> 'OK'").count())

+---------------------+--------------------------+-------------+-------------+----------------+----------------+----------+------+-----------------------------------------------------------+
|                  job|                   dataset|filas_entrada|filas_validas|filas_quarantine|filas_duplicadas|duracion_s|estado|                                                    detalle|
+---------------------+--------------------------+-------------+-------------+----------------+----------------+----------+------+-----------------------------------------------------------+
|         bronze_batch|            customers_orgs|           80|           80|               0|               0|      7.16|    OK|                                      particion=ingest_date|
|         bronze_batch|                     users|          800|          800|               0|               0|      1.58|    OK|                                      particion=ingest_date|
|         bronze_batch|                 resou

Ejecuciones con conciliación fallida: 0


In [25]:
def zona_stats(zona_path):
    filas = []
    for ds in sorted(os.listdir(zona_path)):
        files = glob.glob(f"{zona_path}/{ds}/**/*.parquet", recursive=True)
        parts = {os.path.dirname(f) for f in files}
        filas.append((os.path.basename(zona_path), ds, len(parts), len(files),
                      round(sum(os.path.getsize(f) for f in files) / 1024, 1)))
    return filas

stats = []
for z in ["bronze", "silver", "gold", "quarantine"]:
    stats += zona_stats(P[z])
print(f"{'zona':<12}{'dataset':<32}{'particiones':>12}{'archivos':>10}{'KB':>10}")
for z, ds, p, f, kb in stats:
    print(f"{z:<12}{ds:<32}{p:>12}{f:>10}{kb:>10}")
print("\nEjemplos de rutas:")
for x in sorted(glob.glob(f"{P['gold']}/org_daily_usage_by_service/*"))[:2] + \
         sorted(glob.glob(f"{P['quarantine']}/usage_events_silver/*")):
    print("  ", x)

zona        dataset                          particiones  archivos        KB
bronze      billing_monthly                            3         3      22.3
bronze      customers_orgs                             1         1       7.2
bronze      marketing_touches                          1         1      23.3
bronze      nps_surveys                                1         1       4.2
bronze      resources                                  1         1      12.1
bronze      support_tickets                            1         1      19.3
bronze      usage_events                              60       240    4027.3
bronze      users                                      1         1      24.0
silver      dim_org                                    1         1       7.4
silver      usage_events                              60        60    3236.9
gold        org_daily_usage_by_service                60        60     647.3
gold        top_services_14d                           1         1     110.3

In [26]:
import pandas as pd
from pyspark.sql import Window
ev = P["evidence"]
run_log.orderBy("ts").toPandas().to_csv(f"{ev}/02_run_log.csv", index=False)
pd.DataFrame(stats, columns=["zona", "dataset", "particiones", "archivos", "kb"]).to_csv(f"{ev}/02_lake_stats.csv", index=False)
pd.DataFrame(experimento).to_csv(f"{ev}/02_watermark_experimento.csv", index=False)
(quar_silver.withColumn("rn", F.row_number().over(Window.partitionBy("rule_id").orderBy("event_id")))
    .filter("rn <= 5").drop("rn").toPandas().to_csv(f"{ev}/02_quarantine_muestras.csv", index=False))
anom_pd.to_csv(f"{ev}/02_anomalias_por_grupo.csv", index=False)
mart.orderBy("org_id", "usage_date", "service").limit(200).toPandas().to_csv(f"{ev}/02_gold_muestra.csv", index=False)

resumen = {
    "run_id": RUN_ID, "spark_version": spark.version,
    "streaming": {"bronze": prog_bronze, "watermark": cfg["streaming"]["watermark"],
                  "max_files_per_trigger": cfg["streaming"]["max_files_per_trigger"]},
    "conteos": despues,
    "calidad": {r["rule_id"]: r["count"] for r in quar_silver.groupBy("rule_id").count().collect()},
    "unit_imputadas": silver_ev.filter("unit_imputed").count(),
    "anomalias": {"eventos": tot["a"], "tasa_pct": round(tasa, 2), "metodo": "MAD por (service, metric)",
                  "umbral": cfg["quality"]["anomaly_threshold"]},
    "gold": {"filas": mart.count(), "claves_duplicadas": dup_grano, "costo_silver": cost_silver, "costo_gold": cost_gold},
    "idempotencia": {k: [a, d] for k, a, d, _ in idem},
    "org_ejemplo": ORG_EJEMPLO, "ultima_fecha": str(ultima_fecha),
}
with open(f"{ev}/02_resumen.json", "w") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir(ev)))

['02_anomalias_por_grupo.csv', '02_gold_muestra.csv', '02_lake_stats.csv', '02_quarantine_muestras.csv', '02_resumen.json', '02_run_log.csv', '02_watermark_experimento.csv']


### Limpieza / reinicio

Para volver a correr desde cero (por ejemplo antes de una demo) descomentar y ejecutar: borra todo el lake **menos Landing**, incluidos los checkpoints.

In [27]:
# for z in ["bronze", "silver", "gold", "quarantine", "checkpoints", "meta"]:
#     shutil.rmtree(P[z], ignore_errors=True)
# spark.stop()